In [1]:
import pandas as pd
import os

os.chdir('/Users/ruchapatwardhan/Desktop/IDX Internship')

In [2]:
files = sorted([f for f in os.listdir('raw') if f.startswith('CRMLSListing') and f.endswith('.csv')])

print(f"Files found: {len(files)}")

listings = pd.concat([pd.read_csv(f'raw/{f}') for f in files], ignore_index=True)

print(f"Total rows after concatenation: {listings.shape[0]}")
print(f"Total columns: {listings.shape[1]}")

Files found: 25
Total rows after concatenation: 782537
Total columns: 84


In [3]:
print(f"Rows before Residential filter: {listings.shape[0]}")

listings = listings[listings['PropertyType'] == 'Residential']

print(f"Rows after Residential filter: {listings.shape[0]}")

Rows before Residential filter: 782537
Rows after Residential filter: 494622


In [4]:
listings.to_csv('listings_combined.csv', index=False)
print("Saved listings_combined.csv successfully!")

Saved listings_combined.csv successfully!


In [5]:
files = sorted([f for f in os.listdir('raw') if f.startswith('CRMLSListing') and f.endswith('.csv')])
print(f"Files found: {len(files)}")

listings = pd.concat([pd.read_csv(f'raw/{f}', low_memory=False) for f in files], ignore_index=True)
print(f"Total rows after concatenation: {listings.shape[0]}")
print(f"Total columns: {listings.shape[1]}")

Files found: 25
Total rows after concatenation: 782537
Total columns: 84


In [6]:
print(f"Rows before Residential filter: {listings.shape[0]}")
listings = listings[listings['PropertyType'] == 'Residential']
print(f"Rows after Residential filter: {listings.shape[0]}")

Rows before Residential filter: 782537
Rows after Residential filter: 494622


In [7]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url)
mortgage.columns = ['date', 'rate_30yr_fixed']
mortgage['date'] = pd.to_datetime(mortgage['date'])
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()

listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')
listings = listings.merge(mortgage_monthly, on='year_month', how='left')
print(listings['rate_30yr_fixed'].isnull().sum())

0


In [8]:
listings.to_csv('listings_combined.csv', index=False)
print("Saved listings_combined.csv with mortgage rates!")

Saved listings_combined.csv with mortgage rates!


In [9]:
os.chdir('/Users/ruchapatwardhan/Desktop/IDX Internship')
listings = pd.read_csv('listings_combined.csv', low_memory=False)
print(listings.shape)

(494622, 86)


In [10]:
cols_before = listings.shape[1]
listings = listings.dropna(axis=1, how='all')
cols_after = listings.shape[1]
print(f"Columns before: {cols_before}")
print(f"Columns after: {cols_after}")
print(f"Columns dropped: {cols_before - cols_after}")

Columns before: 86
Columns after: 78
Columns dropped: 8


In [11]:
date_cols = ['ListingContractDate', 'PurchaseContractDate', 'CloseDate', 'ContractStatusChangeDate']
for col in date_cols:
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        print(f"{col} converted successfully")

ListingContractDate converted successfully
PurchaseContractDate converted successfully
CloseDate converted successfully
ContractStatusChangeDate converted successfully


In [12]:
listings['invalid_list_price'] = listings['ListPrice'] <= 0
listings['invalid_living_area'] = listings['LivingArea'] <= 0
listings['missing_coords'] = listings['Latitude'].isnull() | listings['Longitude'].isnull()
listings['wrong_longitude'] = listings['Longitude'] > 0

print(f"Invalid ListPrice: {listings['invalid_list_price'].sum()}")
print(f"Invalid LivingArea: {listings['invalid_living_area'].sum()}")
print(f"Missing coordinates: {listings['missing_coords'].sum()}")
print(f"Wrong longitude: {listings['wrong_longitude'].sum()}")

Invalid ListPrice: 0
Invalid LivingArea: 327
Missing coordinates: 79632
Wrong longitude: 61


In [13]:
listings.to_csv('listings_cleaned.csv', index=False)
print(f"Saved listings_cleaned.csv")
print(f"Final shape: {listings.shape}")

Saved listings_cleaned.csv
Final shape: (494622, 82)
